In [1]:
# ===== 기본 세트 (거의 항상 필요) =====
import pandas as pd                    # 데이터프레임 다루기 (표 형태 데이터 처리)
import numpy as np                     # 수치 계산, 배열 연산
import matplotlib.pyplot as plt        # 기본 시각화
import seaborn as sns                  # matplotlib 기반의, 더 예쁘고 간결한 통계 시각화

# ===== 파일/경로/시스템 =====
import os                              # 파일 경로 확인, 파일 존재 여부(os.path.exists) 등
from pathlib import Path               # 폴더/파일 경로를 객체로 다루기 (work_dir / "파일명")
import joblib                          # 학습된 scaler/encoder를 파일로 저장하고 재사용
import platform                        # OS 종류 확인 (Windows/Mac/Linux 구분 필요할 때)
import warnings                        # 경고 메시지 제어(숨기기/표시)

# ===== 벤치마크/성능 측정 =====
import time                            # 코드 실행 시간 측정 (time.perf_counter())
import psutil                          # 메모리 사용량 측정 (rss_mb 등)
import gc                              # 메모리 강제 정리(garbage collection)

# ===== 결측치 탐색 시각화 =====
import missingno as msno               # 결측치 위치(matrix), 관계(heatmap) 시각화

# ===== 인터랙티브 시각화 =====
import plotly.express as px            # 마우스 hover/zoom 가능한 인터랙티브 그래프
import plotly.io as pio                 # plotly 렌더러(출력 방식) 설정

# ===== 대용량 데이터 처리 =====
import polars as pl                     # pandas보다 빠른 대용량 데이터 처리 (Rust 기반)

# ===== 통계 검증 =====
import statsmodels.api as sm            # 회귀분석, 다중공선성(Cond. No.) 등 통계 검증

# ===== 한글 폰트 설정 =====
import matplotlib.font_manager as fm    # 그래프에 한글 폰트 적용 시 필요

# ===== 인코딩/스케일링/모델링 (sklearn) =====
from sklearn.preprocessing import LabelEncoder      # 카테고리를 숫자로 변환 (순서형)
from sklearn.preprocessing import MinMaxScaler       # 0~1 범위로 스케일링
from sklearn.preprocessing import StandardScaler     # 평균0, 표준편차1로 스케일링
from sklearn.preprocessing import RobustScaler       # 이상치에 강한 스케일링 (median/IQR 기반)
from sklearn.linear_model import LinearRegression    # 선형 회귀 모델
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀(분류) 모델
from sklearn.ensemble import RandomForestRegressor   # 랜덤포레스트(트리 기반) 모델
from sklearn.metrics import roc_auc_score, log_loss  # 분류 모델 평가지표

In [2]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3
저장 폴더: C:\Users\pyostry\workspace\aiffel\ai-data-bootcamp\D009\d009_outputs


In [3]:
# ─────────────────────────────────────────────
# 종합 프로젝트용 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 데이터가 모두 준비됩니다.
# (실제 현장처럼 '오염'을 일부러 심어 둡니다: 결측·이상치·표기 혼재·날짜 포맷 혼재·중복·타입 불일치)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 500
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(36, 10, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
    "signup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(
        np.random.randint(0, 700, n_customers), unit="D"
    ),
})
# 오염 심기: 나이 이상치, 결측, 지역/멤버십 표기 혼재
customers.loc[[5, 6], "age"] = 999
customers.loc[10, "age"] = -3
customers.loc[[20, 21, 22, 23], "gender"] = np.nan
customers.loc[30, "region"] = " 서울 "
customers.loc[31, "region"] = "Seoul"
customers.loc[40, "membership"] = "VIP"
customers.loc[41, "membership"] = " premium"

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 60
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 199900], n_products),
})

# 3) 주문(orders)
n_orders = 5000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app", "app ", "APP"], n_orders, p=[0.45, 0.45, 0.05, 0.05]),
})

# 날짜 포맷 혼재(문자열로 저장 — 일부러 통일하지 않음)
base_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 180, n_orders), unit="D")
date_strings = []
for i, d in enumerate(base_dates):
    if i % 3 == 0:
        date_strings.append(d.strftime("%Y-%m-%d"))
    elif i % 3 == 1:
        date_strings.append(d.strftime("%Y/%m/%d"))
    else:
        date_strings.append(d.strftime("%Y%m%d"))
orders["order_date"] = date_strings

# 오염 심기: 금액 결측, 수량 이상치, 중복 행, 음수 금액(환불로 보이는 값)
orders.loc[np.random.choice(n_orders, 80, replace=False), "amount"] = np.nan
orders.loc[7, "quantity"] = 100
orders.loc[8, "amount"] = -50000.0   # 환불 또는 입력 실수 (판단 필요)
orders = pd.concat([orders, orders.iloc[[0, 1, 2]]], ignore_index=True)  # 중복 3건

# 4) 웹 로그(web_logs) — 텍스트 한 줄로 들어옴(D+005 식 파싱 대상)
n_logs = 1200
log_dates = pd.to_datetime("2025-04-01") + pd.to_timedelta(np.random.randint(0, 60, n_logs), unit="D")
ips = [f"{np.random.randint(1,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}" for _ in range(n_logs)]
paths = np.random.choice(["/home", "/product/123", "/cart", "/checkout", "/search?q=shoes"], n_logs)
web_logs = pd.DataFrame({
    "raw_log": [f"{d.strftime('%Y-%m-%d %H:%M:%S')} - {ip} - GET {p}" for d, ip, p in zip(log_dates, ips, paths)]
})

# CSV로 임시 저장(인제스트 단계에서 읽어옴) — 분석가가 "파일을 받았다" 상황 재현
RAW_DIR = OUT_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)
customers.to_csv(RAW_DIR / "customers.csv", index=False)
products.to_csv(RAW_DIR / "products.csv", index=False)
orders.to_csv(RAW_DIR / "orders.csv", index=False)
web_logs.to_csv(RAW_DIR / "web_logs.csv", index=False)

print("종합 프로젝트 데이터 생성 및 CSV 저장 완료")
print(f"  customers: {customers.shape}")
print(f"  products : {products.shape}")
print(f"  orders   : {orders.shape}  (중복·결측·이상치 포함)")
print(f"  web_logs : {web_logs.shape}")
print(f"\n저장 경로: {RAW_DIR.resolve()}")

종합 프로젝트 데이터 생성 및 CSV 저장 완료
  customers: (500, 6)
  products : (60, 3)
  orders   : (5003, 7)  (중복·결측·이상치 포함)
  web_logs : (1200, 1)

저장 경로: C:\Users\pyostry\workspace\aiffel\ai-data-bootcamp\D009\d009_outputs\raw


In [4]:
# 예제: 방금 저장한 CSV를 다시 읽어 옵니다. (분석가가 새 데이터를 '받은' 상황 재현)
orders_csv = pd.read_csv(RAW_DIR / "orders.csv")
customers_csv = pd.read_csv(RAW_DIR / "customers.csv")
products_csv = pd.read_csv(RAW_DIR / "products.csv")

print("orders_csv.shape :", orders_csv.shape)
print("customers_csv.shape :", customers_csv.shape)
print("products_csv.shape :", products_csv.shape)
print()
print("orders_csv.dtypes")
print(orders_csv.dtypes)

orders_csv.shape : (5003, 7)
customers_csv.shape : (500, 6)
products_csv.shape : (60, 3)

orders_csv.dtypes
order_id           str
customer_id        str
product_id         str
quantity         int64
amount         float64
channel            str
order_date         str
dtype: object


In [5]:
# 예제: customers의 signup_date가 CSV 왕복 후 어떻게 변했는지 확인
print("CSV에서 읽은 직후 dtypes:")
print(customers_csv[["customer_id", "age", "signup_date"]].dtypes)
print()
print("signup_date 첫 5개 값(문자열로 보임):")
print(customers_csv["signup_date"].head().to_list())
print()
# 같은 컬럼을 datetime으로 다시 파싱해야 사용 가능 — 매번 이 작업을 반복해야 합니다.
parsed = pd.to_datetime(customers_csv["signup_date"])
print("파싱 후 dtype:", parsed.dtype)

CSV에서 읽은 직후 dtypes:
customer_id      str
age            int64
signup_date      str
dtype: object

signup_date 첫 5개 값(문자열로 보임):
['2024-07-18', '2025-05-19', '2024-03-27', '2024-02-12', '2025-01-19']

파싱 후 dtype: datetime64[us]


In [6]:
orders.head()

,order_id,customer_id,product_id,quantity,amount,channel,order_date
0,O00001,C0304,P053,3,89700.0,app,2025-03-08
1,O00002,C0276,P016,1,129900.0,web,2025/04/21
2,O00003,C0357,P044,2,19800.0,app,20250608
3,O00004,C0451,P059,2,179800.0,web,2025-05-05
4,O00005,C0172,P054,1,29900.0,app,2025/03/28


In [8]:
products.head()

,product_id,category,price
0,P001,패션,49900
1,P002,패션,89900
2,P003,도서,199900
3,P004,가전,129900
4,P005,식품,199900


In [9]:
customers.head()

,customer_id,age,gender,region,membership,signup_date
0,C0001,41,F,경기,premium,2024-07-18
1,C0002,35,F,경기,basic,2025-05-19
2,C0003,42,F,부산,premium,2024-03-27
3,C0004,51,M,인천,vip,2024-02-12
4,C0005,34,F,인천,basic,2025-01-19


In [10]:
web_logs.head()

,raw_log
0,2025-05-08 00:00:00 - 192.91.141.38 - GET /home
1,2025-05-14 00:00:00 - 195.182.243.41 - GET /cart
2,2025-04-17 00:00:00 - 203.93.229.4 - GET /chec...
3,2025-05-01 00:00:00 - 52.149.204.43 - GET /sea...
4,2025-05-07 00:00:00 - 243.114.212.186 - GET /cart


In [15]:
# 스스로 해보자! (1)
# 1) amount, quantity의 dtype을 출력해보세요.
print(f"amount dtype: {orders['amount'].dtype}")
print(f"quantity dtype: {orders['quantity'].dtype}")
# 2) 결측치(NaN)가 있는 컬럼은 어떤 자료형으로 추론되는지 관찰해보세요.
# 결측치가 있는 컬럼은 ooo64로 기재됨.

amount dtype: float64
quantity dtype: int64


In [16]:
# 예제: 품질 리포트 함수 v1 — 결측·중복·타입을 한 번에 보여주는 진단기
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report


# orders에 적용
qr_orders = quality_report(orders_csv, "orders_csv")
qr_orders

[품질 리포트] orders_csv
  행 수: 5,003  /  열 수: 7
  메모리: 407.5 KB
  완전 중복 행: 3건


,dtype,missing,missing_pct,n_unique,sample
order_id,str,0,0.0,5000,O00001
customer_id,str,0,0.0,500,C0304
product_id,str,0,0.0,60,P053
quantity,int64,0,0.0,4,3
amount,float64,80,1.6,22,89700.0
channel,str,0,0.0,4,app
order_date,str,0,0.0,540,2025-03-08


In [22]:
# 예제: 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base



qr_orders_full = quality_report_full(orders_csv, "orders_csv")
qr_orders_full

[품질 리포트(완전판)] orders_csv
  행 수: 5,003  /  열 수: 7
  완전 중복 행: 3건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
order_id,str,0,0.0,5000,NaN,False
customer_id,str,0,0.0,500,NaN,False
product_id,str,0,0.0,60,NaN,False
quantity,int64,0,0.0,4,0.02,False
amount,float64,80,1.6,22,2.11,False
channel,str,0,0.0,4,NaN,False
order_date,str,0,0.0,540,NaN,False


In [18]:
# 예제: customers와 products에도 같은 함수를 적용 — 한 함수가 어떤 데이터든 진단합니다
print("=" * 60)
qr_customers = quality_report_full(customers_csv, "customers_csv")
display(qr_customers)
print("=" * 60)
qr_products = quality_report_full(products_csv, "products_csv")
display(qr_products)

[품질 리포트(완전판)] customers_csv
  행 수: 500  /  열 수: 6
  완전 중복 행: 0건
  📌 날짜로 보이는 object 컬럼: ['signup_date']


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
customer_id,str,0,0.0,500,NaN,False
age,int64,0,0.0,54,1.4,False
gender,str,4,0.8,2,NaN,False
region,str,0,0.0,7,NaN,False
membership,str,0,0.0,5,NaN,False
signup_date,str,0,0.0,355,NaN,True


[품질 리포트(완전판)] products_csv
  행 수: 60  /  열 수: 3
  완전 중복 행: 0건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
product_id,str,0,0.0,60,NaN,False
category,str,0,0.0,5,NaN,False
price,int64,0,0.0,7,0.0,False


In [19]:
# 스스로 해보자! (2)
def quality_report_with_warning(df, name="df", missing_threshold=30.0):
    base = quality_report_full(df, name)
    base["warning"] = np.where(base["missing_pct"] > missing_threshold, "⚠", "")
    return base

    pass



# quality_report_with_warning(orders_csv, "orders_csv", missing_threshold=10)

SyntaxError: '{' was never closed (1919726572.py, line 6)

In [20]:
def quality_report_with_warning(df, name="df", missing_threshold=30.0):
    base = quality_report_full(df, name)
    base["warning"] = np.where(base["missing_pct"] > missing_threshold, "⚠", "")
    return base

In [40]:
#컬럼별 value_counts()
columns_to_check = ["age", "gender", "region", "membership", "signup_date"]   # 확인하고 싶은 컬럼들

for col in columns_to_check:
    print(f"=== {col} ===")
    print(customers_clean[col].value_counts(dropna = False))
    print()

=== age ===
age
 39     24
 37     23
 31     22
 38     22
 28     22
 41     20
 34     20
 36     20
 35     19
 44     19
 42     17
 30     17
 33     16
 29     16
 40     15
 24     14
 43     14
 27     13
 51     12
 26     12
 22     12
 46     12
 25     10
 45     10
 47      9
 32      7
 21      6
 50      6
 52      6
 48      6
 55      5
 16      5
 23      5
 20      5
 49      5
 57      5
 17      3
 19      3
 58      3
 54      3
 999     2
 18      2
 15      2
-3       1
 10      1
 61      1
 63      1
 75      1
 59      1
 4       1
 56      1
 13      1
 67      1
 53      1
Name: count, dtype: int64

=== gender ===
gender
M      259
F      237
NaN      4
Name: count, dtype: int64

=== region ===
region
경기       105
서울       104
대구       103
부산        99
인천        87
 서울        1
Seoul      1
Name: count, dtype: int64

=== membership ===
membership
basic       307
premium     153
vip          38
VIP           1
 premium      1
Name: count, dtype: int64

=== 

In [59]:
#결측 확인
customers_clean[["age", "gender", "region", "membership", "signup_date"]].isna().sum()

age            0
gender         4
region         0
membership     0
signup_date    0
dtype: int64

In [61]:
customers_clean[customers_clean["gender"].isna()]

,customer_id,age,gender,region,membership,signup_date,age_was_missing
20,C0021,51,NaN,부산,basic,2024-07-29,False
21,C0022,34,NaN,서울,basic,2025-01-25,False
22,C0023,37,NaN,서울,basic,2025-05-13,False
23,C0024,22,NaN,경기,basic,2025-06-23,False


In [69]:
# 스스로 해보자! (3)
customers_clean = customers_csv.copy()
# 1. 중복 제거
def duplicated(customers_clean):
    customers_clean = customers_clean.drop_duplicates(keep = "first")
    return customers_clean

customers_clean = duplicated(customers_clean)

# 2. 이상치 처리
#age
def age_clean(customers_clean):
    customers_clean["age"] = customers_clean["age"].where(
        (customers_clean["age"] >= 0) & (customers_clean["age"] <= 120), 
        np.nan)
    customers_clean["age"] = customers_clean["age"].astype("Int64")
    return customers_clean

customers_clean = age_clean(customers_clean)

# 3. 결측치 처리
#age
def age_na_clean(customers_clean):
    customers_clean["age_was_missing"] = customers_clean["age"].isna()
    customers_clean["age"] = customers_clean["age"].fillna(customers_clean["age"].mean().round(0))
    return customers_clean

customers_clean = age_na_clean(customers_clean)

# 4. 문자열 정제
#region
def region_clean(customers_clean):
    customers_clean["region"] = customers_clean["region"].str.strip().str.lower()
    customers_clean["region"] = customers_clean["region"].str.replace({"Seoul" : "서울"})
    return customers_clean

customers_clean = region_clean(customers_clean)

# membership
def membership_clean(customers_clean):
    customers_clean["membership"] = customers_clean["membership"].str.strip().str.lower()
    return customers_clean

customers_clean = membership_clean(customers_clean)


# 5. 날짜 파싱
def signup_date_parsing(customers_clean):
    customers_clean["signup_date"] = pd.to_datetime(customers_clean["signup_date"], format = "mixed")
    return customers_clean

customers_clean = signup_date_parsing(customers_clean)



print(customers_clean.shape)

(500, 7)


In [72]:
customers_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   customer_id      500 non-null    str           
 1   age              500 non-null    Int64         
 2   gender           496 non-null    str           
 3   region           500 non-null    str           
 4   membership       500 non-null    str           
 5   signup_date      500 non-null    datetime64[us]
 6   age_was_missing  500 non-null    bool          
dtypes: Int64(1), bool(1), datetime64[us](1), str(4)
memory usage: 33.2 KB
